# P2 프로젝트: 자전거 대여 수요 예측

P2 실습에서는 이진 분류, 다중 클래스 분류, 회귀 문제를 다루고
훈련·검증·테스트 데이터의 역할과 일반화 성능을 확인했다.

이번 프로젝트에서는 **시간별 자전거 대여 수요를 예측하는 회귀 문제**를 해결한다.

프로젝트의 핵심은 단순히 코드를 실행하는 것이 아니라,

> **문제를 이해하고 → 입력과 타깃을 확인하고 → 평가 방법을 결정하고 → 모델을 설계하고 → 일반화 성능을 판단하는 것**

이다.

## 프로젝트에서 해야 할 일

1. 데이터와 타깃의 의미를 확인한다.
2. 예측에 사용하면 안 되는 변수가 있는지 판단한다.
3. 시간 순서를 고려하여 훈련셋·검증셋·테스트셋을 구성한다.
4. 회귀 문제에 적절한 출력층, 손실함수, 평가지표를 선택한다.
5. **Keras와 PyTorch에서 모두** 기준 모델을 구성하고 훈련한다.
6. 두 프레임워크에서 훈련·검증 성능과 일반화 상태를 비교한다.
7. 각 프레임워크에서 일반화 성능을 개선하기 위한 방법을 하나 이상 적용한다.
8. Keras와 PyTorch의 최종 모델을 각각 테스트 데이터에서 평가한다.
9. 실제 대여량과 예측 대여량을 비교하고 오차가 큰 사례를 분석한다.

## 생성형 AI 활용

생성형 AI는 코드 작성·수정, 오류 메시지 해석, Keras와 PyTorch 코드 변환,
그래프 작성, 모델 개선 아이디어 제안 등에 활용할 수 있다.

다만 다음은 직접 확인하고 판단해야 한다.

- 어떤 변수를 입력으로 사용할 것인가?
- 어떤 변수가 데이터 누수를 일으킬 수 있는가?
- 왜 시간 순서를 고려하여 데이터를 나누는가?
- 어떤 모델 구조와 일반화 방법을 사용할 것인가?
- Keras와 PyTorch에서 같은 회귀 문제를 어떻게 표현하는가?
- 두 프레임워크의 결과를 어떻게 비교하고 해석할 것인가?
- 결과와 오류를 어떻게 해석할 것인가?

프로젝트 마지막에 **AI를 어디에 활용했고 무엇을 직접 검증했는지** 기록한다.

---

## 실행 환경 확인

In [ ]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import tensorflow as tf
import keras
import torch

print("TensorFlow:", tf.__version__)
print("Keras:", keras.__version__)
print("PyTorch:", torch.__version__)
print("TensorFlow GPU:", tf.config.list_physical_devices("GPU"))
print("PyTorch CUDA:", torch.cuda.is_available())

---

## 데이터 불러오기

UCI Bike Sharing 데이터셋의 **시간별 데이터(hour.csv)** 를 사용한다.
이 데이터에는 2011~2012년의 시간별 자전거 대여량과
날씨·계절·시간 정보가 포함되어 있다.

데이터 불러오기와 기본 전처리 코드는 제공한다.
프로젝트의 중심은 데이터 다운로드 방법이 아니라 **회귀 모델의 설계와 평가**다.

In [ ]:
from pathlib import Path
import zipfile
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

url = "https://archive.ics.uci.edu/static/public/275/bike+sharing+dataset.zip"

archive_path = keras.utils.get_file(
    fname="bike_sharing_dataset.zip",
    origin=url,
)

data_dir = Path(archive_path).parent / "bike_sharing_dataset"
data_dir.mkdir(exist_ok=True)

with zipfile.ZipFile(archive_path) as z:
    z.extractall(data_dir)

hour_path = next(data_dir.rglob("hour.csv"))
bike = pd.read_csv(hour_path)

bike.head()

### 데이터 변수

주요 변수는 다음과 같다.

| 변수 | 의미 |
|---|---|
| `dteday` | 날짜 |
| `season` | 계절 |
| `yr` | 연도 |
| `mnth` | 월 |
| `hr` | 시간 |
| `holiday` | 공휴일 여부 |
| `weekday` | 요일 |
| `workingday` | 근무일 여부 |
| `weathersit` | 날씨 상태 |
| `temp` | 정규화된 기온 |
| `atemp` | 정규화된 체감온도 |
| `hum` | 정규화된 습도 |
| `windspeed` | 정규화된 풍속 |
| `casual` | 비등록 이용자의 대여 건수 |
| `registered` | 등록 이용자의 대여 건수 |
| `cnt` | 전체 자전거 대여 건수 |

이번 프로젝트의 타깃은 `cnt`다.

In [ ]:
print("데이터 크기:", bike.shape)
print()
print(bike.info())

### 문제 1 — 이 문제는 왜 회귀 문제인가?

다음 질문에 답하라.

1. 입력은 무엇인가?
2. 타깃은 무엇인가?
3. `cnt`는 정수인데도 왜 분류보다 회귀로 다루는 것이 자연스러운가?
4. 어떤 손실함수와 평가지표를 사용할 수 있는가?

**답:**

> 여기에 작성한다.

---

## 데이터 누수 확인

원 데이터에는 다음 관계가 있다.

$$
	ext{cnt} = 	ext{casual} + 	ext{registered}
$$

따라서 `casual`과 `registered`를 입력으로 사용하면
모델이 예측해야 할 정답을 사실상 입력에서 직접 알 수 있게 된다.

이처럼 예측 시점에는 사용할 수 없어야 할 정보가 모델 입력에 포함되는 현상을
**데이터 누수**(data leakage)라고 한다.

### 문제 2 — 어떤 변수를 제외해야 하는가?

다음 변수 중 모델 입력에서 제외할 변수를 결정하고 이유를 설명하라.

- `instant`
- `dteday`
- `casual`
- `registered`

**답:**

> 여기에 작성한다.

---

## 시간 순서를 고려한 데이터 분할

자전거 대여 데이터에는 **시간의 순서**가 있다.

실제 예측 상황에서는 과거의 데이터를 이용하여 이후의 수요를 예측한다.
따라서 여기서는 데이터를 무작위로 섞지 않고 시간 순서대로 나눈다.

> **앞 70%: 훈련셋 → 다음 15%: 검증셋 → 마지막 15%: 테스트셋**

테스트셋은 최종 평가 전까지 사용하지 않는다.

In [ ]:
bike["datetime"] = pd.to_datetime(bike["dteday"]) + pd.to_timedelta(bike["hr"], unit="h")
bike = bike.sort_values("datetime").reset_index(drop=True)

n = len(bike)
train_end = int(n * 0.70)
val_end = int(n * 0.85)

train_df = bike.iloc[:train_end].copy()
val_df = bike.iloc[train_end:val_end].copy()
test_df = bike.iloc[val_end:].copy()

print("train:", train_df.shape)
print("validation:", val_df.shape)
print("test:", test_df.shape)

print()
print("train 기간:", train_df["datetime"].min(), "~", train_df["datetime"].max())
print("validation 기간:", val_df["datetime"].min(), "~", val_df["datetime"].max())
print("test 기간:", test_df["datetime"].min(), "~", test_df["datetime"].max())

### 문제 3 — 왜 무작위 분할을 사용하지 않았는가?

1. 이 데이터를 무작위로 섞어 나누면 어떤 문제가 생길 수 있는가?
2. 일반적인 무작위 K-fold 교차 검증을 그대로 적용하면 왜 실제 예측 상황과 맞지 않을 수 있는가?
3. 이번 프로젝트의 시간순 분할이 어떤 실제 상황을 모방하는지 설명하라.

**답:**

> 여기에 작성한다.

---

## 입력 데이터 준비

전처리 자체가 프로젝트의 중심이 되지 않도록 기본 코드를 제공한다.

- `casual`, `registered`, `cnt`, `instant`, 날짜 관련 원본 변수는 입력에서 제외한다.
- 범주형 변수는 one-hot encoding한다.
- 연속형 변수는 훈련셋의 기준으로 표준화한다.
- 검증셋과 테스트셋에는 훈련셋에서 학습한 변환을 그대로 적용한다.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

target = "cnt"

categorical_features = [
    "season", "yr", "mnth", "hr",
    "holiday", "weekday", "workingday", "weathersit"
]

numeric_features = [
    "temp", "atemp", "hum", "windspeed"
]

feature_columns = categorical_features + numeric_features

X_train_raw = train_df[feature_columns]
X_val_raw = val_df[feature_columns]
X_test_raw = test_df[feature_columns]

y_train = train_df[target].to_numpy(dtype="float32")
y_val = val_df[target].to_numpy(dtype="float32")
y_test = test_df[target].to_numpy(dtype="float32")

preprocessor = ColumnTransformer([
    ("categorical", OneHotEncoder(handle_unknown="ignore", sparse_output=False),
     categorical_features),
    ("numeric", StandardScaler(), numeric_features),
])

X_train = preprocessor.fit_transform(X_train_raw).astype("float32")
X_val = preprocessor.transform(X_val_raw).astype("float32")
X_test = preprocessor.transform(X_test_raw).astype("float32")

print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("X_test:", X_test.shape)

### 전처리 확인

1. one-hot encoding이 필요한 변수들은 어떤 성격의 변수인가?
2. 표준화 기준을 훈련셋에서만 계산해야 하는 이유는 무엇인가?
3. `X_train.shape[1]`이 원래 입력 변수의 개수보다 커진 이유는 무엇인가?

**답:**

> 여기에 작성한다.

---

## 모델링

이번 프로젝트에서는 **Keras와 PyTorch를 모두 사용한다.**

가능하면 두 프레임워크에서 은닉층 수와 유닛 수 등
모델 구조를 비슷하게 구성하여 결과를 비교한다.

기준 모델은 P2 실습의 California Housing 회귀 모델을 참고할 수 있지만,
**은닉층 수와 유닛 수는 직접 결정한다.**

모델을 지나치게 복잡하게 만들기보다,
두 프레임워크에서 같은 회귀 원리가 어떻게 구현되는지 확인하고
기준 모델과 개선 모델의 차이를 설명할 수 있도록 구성한다.

### 공통 모델 설계 원칙

Keras와 PyTorch에서 가능한 한 비슷한 구조를 사용한다.

- 은닉층 수:
- 각 은닉층의 유닛 수:
- 활성화 함수:
- 출력층:
- 손실함수:
- 평가지표:
- batch size:
- 최대 epoch:

**이 구조를 선택한 이유:**

> 여기에 작성한다.

### Keras와 PyTorch에서 확인할 점

같은 회귀 문제를 두 프레임워크에서 구현하면서 다음을 비교한다.

- 모델의 마지막 출력이 어떻게 구성되는가?
- MSE 손실과 RMSE 평가지표는 각각 어떻게 지정하는가?
- 훈련 과정은 어떤 부분이 자동화되어 있고 어떤 부분을 직접 작성하는가?
- 같은 구조를 사용해도 결과가 완전히 같지 않을 수 있는 이유는 무엇인가?

> 비교하면서 발견한 차이를 간단히 기록한다.

---

## Keras 모델

먼저 Keras에서 기준 모델과 개선 모델을 구성하고 훈련한다.

### Keras 기준 모델

In [ ]:
from keras import layers

# TODO: 기준 모델을 구성한다.
keras_base = keras.Sequential([
    # layers.Dense(..., activation="relu"),
    # ...
    # layers.Dense(1)
])

# TODO: 입력 shape에 맞게 모델을 build한다.
# keras_base.build(input_shape=(None, X_train.shape[1]))
# keras_base.summary()

In [ ]:
# TODO: 모델을 compile하고 훈련한다.
#
# keras_base.compile(
#     optimizer="adam",
#     loss="mse",
#     metrics=[keras.metrics.RootMeanSquaredError(name="rmse")]
# )
#
# history_base = keras_base.fit(
#     X_train,
#     y_train,
#     epochs=...,
#     batch_size=...,
#     validation_data=(X_val, y_val),
#     verbose=1,
# )

### Keras 개선 모델

기준 모델의 검증 결과를 확인한 뒤
다음 방법 중 하나 이상을 선택하여 개선 모델을 만든다.

- 모델 크기 변경
- L2 가중치 규제
- Dropout
- EarlyStopping

**변경 내용과 이유:**

> 여기에 작성한다.

In [ ]:
# TODO: 개선 모델과 필요한 callback을 구성한다.
#
# improved_model = keras.Sequential([
#     ...
# ])
#
# improved_model.compile(...)
#
# history_improved = improved_model.fit(
#     X_train,
#     y_train,
#     validation_data=(X_val, y_val),
#     ...
# )

---

## PyTorch 모델

같은 문제를 PyTorch에서도 구현한다.
가능하면 Keras 모델과 비슷한 은닉층 구조를 사용한다.

In [ ]:
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

train_dataset = TensorDataset(
    torch.tensor(X_train, dtype=torch.float32),
    torch.tensor(y_train, dtype=torch.float32).reshape(-1, 1)
)
val_dataset = TensorDataset(
    torch.tensor(X_val, dtype=torch.float32),
    torch.tensor(y_val, dtype=torch.float32).reshape(-1, 1)
)
test_dataset = TensorDataset(
    torch.tensor(X_test, dtype=torch.float32),
    torch.tensor(y_test, dtype=torch.float32).reshape(-1, 1)
)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=256, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False)

print("device:", device)

### PyTorch 훈련 함수

프로젝트의 중심은 반복문의 문법이 아니라
**모델을 비교하고 결과를 판단하는 것**이므로 기본 훈련 함수를 제공한다.

In [ ]:
def train_regression_model(model, train_loader, val_loader, epochs=50,
                           weight_decay=0.0, patience=None):
    model = model.to(device)
    loss_fn = nn.MSELoss()
    optimizer = torch.optim.Adam(
        model.parameters(),
        weight_decay=weight_decay
    )

    history = {
        "loss": [], "val_loss": [],
        "rmse": [], "val_rmse": []
    }

    best_val_loss = float("inf")
    best_state = None
    wait = 0

    for epoch in range(epochs):
        model.train()
        train_sse = 0.0
        train_n = 0

        for x_batch, y_batch in train_loader:
            x_batch = x_batch.to(device)
            y_batch = y_batch.to(device)

            optimizer.zero_grad()
            pred = model(x_batch)
            loss = loss_fn(pred, y_batch)
            loss.backward()
            optimizer.step()

            train_sse += ((pred - y_batch) ** 2).sum().item()
            train_n += len(x_batch)

        model.eval()
        val_sse = 0.0
        val_n = 0

        with torch.no_grad():
            for x_batch, y_batch in val_loader:
                x_batch = x_batch.to(device)
                y_batch = y_batch.to(device)

                pred = model(x_batch)
                val_sse += ((pred - y_batch) ** 2).sum().item()
                val_n += len(x_batch)

        train_mse = train_sse / train_n
        val_mse = val_sse / val_n

        history["loss"].append(train_mse)
        history["val_loss"].append(val_mse)
        history["rmse"].append(np.sqrt(train_mse))
        history["val_rmse"].append(np.sqrt(val_mse))

        if patience is not None:
            if val_mse < best_val_loss:
                best_val_loss = val_mse
                best_state = {
                    k: v.detach().cpu().clone()
                    for k, v in model.state_dict().items()
                }
                wait = 0
            else:
                wait += 1
                if wait >= patience:
                    break

    if best_state is not None:
        model.load_state_dict(best_state)

    return model, history

### PyTorch 기준 모델

In [ ]:
# TODO: 기준 모델을 구성한다.
torch_base = nn.Sequential(
    # nn.Linear(X_train.shape[1], ...),
    # nn.ReLU(),
    # ...
    # nn.Linear(..., 1),
)

# TODO: 기준 모델을 훈련한다.
# torch_base, torch_history_base = train_regression_model(
#     torch_base,
#     train_loader,
#     val_loader,
#     epochs=...
# )

### PyTorch 개선 모델

**변경 내용과 이유:**

> 여기에 작성한다.

In [ ]:
# TODO: 개선 모델을 구성하고 훈련한다.
#
# torch_improved = nn.Sequential(
#     ...
# )
#
# torch_improved, torch_history_improved = train_regression_model(
#     torch_improved,
#     train_loader,
#     val_loader,
#     epochs=...,
#     weight_decay=...,
#     patience=...
# )

---

## 훈련 과정과 일반화 성능 비교

**Keras와 PyTorch 모두에서** 기준 모델과 개선 모델의
훈련 RMSE와 검증 RMSE를 확인한다.

먼저 각 프레임워크 안에서 기준 모델과 개선 모델을 비교하고,
그 다음 두 프레임워크의 결과를 비교한다.

In [ ]:
# Keras
keras_base_history = history_base.history
keras_improved_history = history_improved.history

# PyTorch
pytorch_base_history = torch_history_base
pytorch_improved_history = torch_history_improved

In [ ]:
# TODO: 훈련·검증 RMSE를 시각화한다.
#
# plt.figure(figsize=(7, 4))
# plt.plot(base_history["rmse"], label="Base training RMSE")
# plt.plot(base_history["val_rmse"], label="Base validation RMSE")
# plt.xlabel("Epoch")
# plt.ylabel("RMSE")
# plt.legend()
# plt.show()
#
# 개선 모델도 같은 방식으로 확인한다.

### 결과 비교와 모델 선택

다음 질문에 답한다.

1. Keras 기준 모델에서 과대적합이 나타났는가?
2. PyTorch 기준 모델에서 과대적합이 나타났는가?
3. 각 프레임워크에서 일반화 성능을 개선하기 위해 무엇을 변경했는가?
4. 개선 후 검증 RMSE는 어떻게 달라졌는가?
5. Keras와 PyTorch의 검증 성능은 얼마나 비슷하거나 달랐는가?
6. 같은 구조를 사용했는데도 결과가 다르다면 가능한 이유는 무엇인가?
7. 각 프레임워크에서 최종 모델로 어떤 모델을 선택하겠는가?
8. 단순히 훈련 RMSE가 가장 낮은 모델을 선택하면 안 되는 이유는 무엇인가?

**답:**

> 여기에 비교와 판단을 작성한다.

---

## 최종 테스트 평가

모델 선택이 끝난 뒤 **Keras와 PyTorch에서 선택한 최종 모델을 각각**
테스트셋에서 한 번 평가한다.

테스트 결과를 보고 다시 모델을 수정하지 않는다.

In [ ]:
# Keras 최종 모델
keras_final_model = ...

test_mse, keras_test_rmse = keras_final_model.evaluate(
    X_test, y_test, verbose=0
)

print("Keras test MSE :", test_mse)
print("Keras test RMSE:", keras_test_rmse)

In [ ]:
# PyTorch 최종 모델
pytorch_final_model = ...
pytorch_final_model.eval()

predictions = []

with torch.no_grad():
    for x_batch, _ in test_loader:
        pred = pytorch_final_model(x_batch.to(device))
        predictions.append(pred.cpu().numpy())

pytorch_predictions = np.concatenate(predictions).reshape(-1)
pytorch_test_rmse = np.sqrt(np.mean((pytorch_predictions - y_test) ** 2))

print("PyTorch test RMSE:", pytorch_test_rmse)

---

## 실제값과 예측값 비교

최종 모델의 예측값을 이용하여
실제 자전거 대여량과 예측 대여량을 비교한다.

In [ ]:
# TODO: 최종 모델의 테스트 예측값을 final_predictions에 저장한다.
#
# Keras 예:
# final_predictions = final_model.predict(X_test, verbose=0).reshape(-1)
#
# PyTorch에서는 위 테스트 평가에서 만든 predictions를 사용할 수 있다.

In [ ]:
# TODO: 실제값과 예측값 일부를 비교한다.
#
# comparison = pd.DataFrame({
#     "actual": y_test,
#     "predicted": final_predictions,
# })
#
# comparison["error"] = comparison["actual"] - comparison["predicted"]
# comparison["abs_error"] = comparison["error"].abs()
#
# comparison.head(20)

### 오차 분석

1. 절대오차가 큰 사례를 10개 정도 찾아본다.
2. 어떤 시간대나 날씨 조건에서 큰 오차가 나타나는지 확인한다.
3. 대여량이 매우 높은 시점과 낮은 시점 중 어느 쪽에서 예측이 더 어려운가?
4. 현재 입력 변수만으로 설명하기 어려운 요인은 무엇이 있을까?

**답:**

> 여기에 분석을 작성한다.

---

## 프로젝트 최종 정리

### 문제와 데이터
> 무엇을 예측했고 입력과 타깃은 무엇이었는가?

### 데이터 누수
> 제외한 변수와 그 이유는 무엇인가?

### 평가 설계
> 데이터를 어떻게 나누었고 왜 시간 순서를 고려했는가?

### 공통 모델 설계
> Keras와 PyTorch에서 어떤 구조, 손실함수, 평가지표를 사용했는가?

### 일반화
> 각 프레임워크에서 과대적합이 있었는가? 어떻게 판단했고 어떤 방법으로 대응했는가?

### Keras와 PyTorch 비교
> 두 프레임워크의 구현 방식과 검증 성능에는 어떤 공통점과 차이가 있었는가?

### 최종 결과
> Keras와 PyTorch의 테스트 RMSE는 각각 얼마이며, 어떻게 해석할 수 있는가?

### 오류와 한계
> 어떤 상황에서 오차가 컸으며 현재 모델의 한계는 무엇인가?

### 생성형 AI 활용과 검증
> AI를 어디에 활용했고, 어떤 결과를 직접 확인·검증했는가?

## 제출 전 확인

- [ ] 회귀 문제와 타깃의 의미를 설명했다.
- [ ] 데이터 누수를 일으킬 수 있는 변수를 확인했다.
- [ ] 시간 순서를 고려하여 데이터를 분할했다.
- [ ] 모델 구조와 손실함수·평가지표의 선택 이유를 설명했다.
- [ ] **Keras 기준 모델과 개선 모델을 모두 훈련했다.**
- [ ] **PyTorch 기준 모델과 개선 모델을 모두 훈련했다.**
- [ ] 두 프레임워크에서 훈련 성능과 검증 성능을 비교했다.
- [ ] Keras와 PyTorch의 구현 방식과 결과를 비교했다.
- [ ] 각 프레임워크의 최종 모델을 선택하고 근거를 설명했다.
- [ ] 테스트셋은 각 최종 모델의 마지막 평가에만 사용했다.
- [ ] 실제값과 예측값을 비교하고 오류를 분석했다.
- [ ] 생성형 AI 활용 내용과 검증 방법을 기록했다.